# Notebook 2 — Extract Entities & Relationships with Azure OpenAI

This is the **second step** of the GraphRAG pipeline. It reads each Wikipedia article produced in Notebook 1 and uses an **Azure OpenAI GPT-4o** model to extract structured knowledge from the free text.

## What this notebook does
1. Loads each `.txt` article from `data/wikipedia/`
2. Sends the full article to GPT-4o with a structured extraction prompt (`prompts/findentites.md`)
3. The model returns a JSON object with two keys:
   - **`entities`** — named things: `{ name, type, description }` (e.g. `Marie Curie | Person | …`)
   - **`relationships`** — typed triples: `{ source, relationship, target, evidence, confidence }` (e.g. `Marie Curie COLLABORATED_WITH Pierre Curie`)
4. Saves the JSON response to `data/wikipedia_entities/<PageTitle>.txt`

Already-processed files are skipped automatically.

## Relationship types
The extraction prompt enforces a strict vocabulary of relationship types (e.g. `DISCOVERED`, `EDUCATED_AT`, `INFLUENCED`, `COLLABORATED_WITH`) to keep the knowledge graph consistent across all articles.

## Output
`data/wikipedia_entities/` — 51 JSON files containing entities and relationships extracted from each article.

## Next step
Run **Notebook 3** to import these JSON triples into Neo4j as a property graph.

## 1 · Imports

In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI

## 2 · Configuration

In [6]:
SCRIPT_DIR = Path(".").resolve()
load_dotenv(dotenv_path=SCRIPT_DIR.parent / ".env")

PROMPT_TEMPLATE = (SCRIPT_DIR.parent / "prompts" / "findentites.md").read_text(encoding="utf-8")
WIKIPEDIA_DIR   = SCRIPT_DIR.parent / "data" / "wikipedia"
OUTPUT_DIR      = SCRIPT_DIR.parent / "data" / "wikipedia_entities"

endpoint       = os.getenv("endpoint")
subscription_key = os.getenv("subscription_key")
deployment     = os.getenv("deployment")
api_version    = os.getenv("api_version")

print(f"Wikipedia dir  : {WIKIPEDIA_DIR}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Deployment     : {deployment}")

Wikipedia dir  : C:\Repos\GraphRAG_Demo\src\data\wikipedia
Output dir     : C:\Repos\GraphRAG_Demo\src\data\wikipedia_entities
Deployment     : gpt-4o


## 3 · Initialise Azure OpenAI Client

In [7]:
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
    azure_deployment=deployment,
)

print("Client initialised.")

Client initialised.


## 4 · Extract Triples from Each Article

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

wikipedia_files = sorted(WIKIPEDIA_DIR.glob("*.txt"))
print(f"Found {len(wikipedia_files)} Wikipedia files to process.\n")

for wiki_file in wikipedia_files:
    out_file = OUTPUT_DIR / wiki_file.name

    if out_file.exists():
        print(f"--- Skipping (already done): {wiki_file.name} ---")
        continue

    print(f"--- Processing: {wiki_file.name} ---")
    wikipedia_text = wiki_file.read_text(encoding="utf-8")

    prompt = PROMPT_TEMPLATE.replace("{{wikipedia_text_chunk}}", wikipedia_text)
    messages = [{"role": "user", "content": prompt}]

    response = client.chat.completions.create(
        messages=messages,
        temperature=1.0,
        top_p=1.0,
        model=deployment,
    )
    result = response.choices[0].message.content

    out_file.write_text(result, encoding="utf-8")
    print(f"  [saved] {out_file.name}\n")

print("Done.")

Found 51 Wikipedia files to process.

--- Processing: Ada_Lovelace.txt ---
  [saved] Ada_Lovelace.txt

--- Processing: Al-Khwarizmi.txt ---
  [saved] Al-Khwarizmi.txt

--- Processing: Alan_Turing.txt ---
  [saved] Alan_Turing.txt

--- Processing: Albert_Einstein.txt ---
  [saved] Albert_Einstein.txt

--- Processing: Alexander_Fleming.txt ---
  [saved] Alexander_Fleming.txt

--- Processing: Antoine_Lavoisier.txt ---
  [saved] Antoine_Lavoisier.txt

--- Processing: Archimedes.txt ---
  [saved] Archimedes.txt

--- Processing: Barbara_McClintock.txt ---
  [saved] Barbara_McClintock.txt

--- Processing: Carl_Sagan.txt ---
  [saved] Carl_Sagan.txt

--- Processing: Caroline_Herschel.txt ---
  [saved] Caroline_Herschel.txt

--- Processing: Cecilia_Payne-Gaposchkin.txt ---
  [saved] Cecilia_Payne-Gaposchkin.txt

--- Processing: Charles_Darwin.txt ---
  [saved] Charles_Darwin.txt

--- Processing: Chien-Shiung_Wu.txt ---
  [saved] Chien-Shiung_Wu.txt

--- Processing: Claude_Shannon.txt ---
  [sav